## Tahap 3 — Kalkulasi RFM

**RFM** adalah tiga metrik yang mengukur perilaku pembelian pelanggan:

| Metrik | Definisi | Semakin tinggi = |
|--------|----------|------------------|
| **Recency (R)** | Berapa hari sejak transaksi terakhir? | Semakin *rendah* nilainya semakin baik |
| **Frequency (F)** | Berapa kali pelanggan bertransaksi? | Semakin tinggi semakin setia |
| **Monetary (M)** | Berapa total uang yang dibelanjakan? | Semakin tinggi semakin berharga |

skor 1–5 untuk masing-masing (5 = terbaik)

In [2]:
import pandas as pd
import numpy as np

df_clean = pd.read_csv('../data/data_clean.csv', parse_dates=['InvoiceDate'])

# Tanggal referensi = 1 hari setelah transaksi terakhir
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Snapshot date: {snapshot_date}')

rfm = df_clean.groupby('Customer ID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('Invoice',     'nunique'),
    Monetary  = ('TotalPrice',  'sum')
).reset_index()

print(f'Jumlah pelanggan: {len(rfm):,}')
rfm.describe().round(2)

Snapshot date: 2011-12-10 12:50:00
Jumlah pelanggan: 5,861


,Customer ID,Recency,Frequency,Monetary
count,5861.00,5861.00,5861.00,5861.00
mean,15317.03,200.94,6.25,2912.96
std,1715.98,209.20,12.77,14300.36
min,12346.00,1.00,1.00,2.95
25%,13834.00,26.00,1.00,338.13
50%,15318.00,96.00,3.00,854.99
75%,16800.00,379.00,7.00,2237.12
max,18287.00,739.00,376.00,580987.04


In [3]:
# Siapa pelanggan dengan Monetary tertinggi?
print(rfm.nlargest(5, 'Monetary')[['Customer ID', 'Recency', 'Frequency', 'Monetary']])

# Distribusi Monetary
print(f'\nPelanggan dengan Monetary > £10.000: {(rfm["Monetary"] > 10000).sum()}')
print(f'Persentase: {(rfm["Monetary"] > 10000).sum() / len(rfm) * 100:.1f}%')

      Customer ID  Recency  Frequency   Monetary
5675        18102        1        145  580987.04
2269        14646        2        145  526751.52
1784        14156       10        150  303609.17
2527        14911        1        376  272801.15
5035        17450        8         51  244784.25

Pelanggan dengan Monetary > £10.000: 254
Persentase: 4.3%


In [5]:
# Cap Monetary di persentil 99
p99 = rfm['Monetary'].quantile(0.99)
print(f'Persentil 99 Monetary : £{p99:,.0f}')
print(f'Pelanggan yang di-cap : {(rfm["Monetary"] > p99).sum()}')
rfm['Monetary_capped'] = rfm['Monetary'].clip(upper=p99)

# Scoring 1–5
rfm['R_score'] = pd.qcut(rfm['Recency'], q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['Monetary_capped'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)

rfm['RFM_Score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
rfm['RFM_Total'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

# Segmentasi
def assign_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

# Ringkasan
summary = rfm.groupby('Segment').agg(
    Jumlah_Pelanggan = ('Customer ID', 'count'),
    Avg_Recency      = ('Recency',     'mean'),
    Avg_Frequency    = ('Frequency',   'mean'),
    Avg_Monetary     = ('Monetary',    'mean'),
    Total_Revenue    = ('Monetary',    'sum')
).round(1)
summary['Pct_Pelanggan'] = (summary['Jumlah_Pelanggan'] / len(rfm) * 100).round(1)
summary['Pct_Revenue']   = (summary['Total_Revenue'] / rfm['Monetary'].sum() * 100).round(1)
print(summary.to_string())

# Bulatkan Monetary untuk kompatibilitas Power BI
rfm['Monetary'] = rfm['Monetary'].round(0).astype(int)
rfm['Monetary_capped'] = rfm['Monetary_capped'].round(0).astype(int)

rfm.to_csv('../data/rfm_result_pbi.csv', index=False)
print('\nHasil disimpan ke rfm_result.csv')

Persentil 99 Monetary : £28,616
Pelanggan yang di-cap : 59
                 Jumlah_Pelanggan  Avg_Recency  Avg_Frequency  Avg_Monetary  Total_Revenue  Pct_Pelanggan  Pct_Revenue
Segment                                                                                                               
At Risk                       828        369.3            5.0        1919.4        1589286           14.1          9.3
Champions                    1293         20.0           17.0        9018.2       11660589           22.1         68.3
Lost                         1894        387.5            1.3         432.6         819267           32.3          4.8
Loyal Customers              1395         71.3            5.4        1854.7        2587286           23.8         15.2
New Customers                 451         28.1            1.5         923.3         416406            7.7          2.4

Hasil disimpan ke rfm_result.csv
